# Ćwiczenia 1: Apache Kafka — producent, konsument, reguły decyzyjne

## Cel ćwiczenia:
- Uruchomienie Kafki, utworzenie tematu
- Napisanie producenta i konsumenta w języku Python
- Filtrowanie (stateless) i zliczanie (stateful) zdarzeń
- Implementacja prostych reguł decyzyjnych na żywym strumieniu

# Część 1: Producent (20 min)

Poniżej gotowy producent — przeanalizuj kod, uruchom w terminalu (python producer.py).

In [2]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


## Zadanie 1.1 — Zmodyfikuj producenta

Zmień producenta tak, by 5% transakcji było podejrzanych:
- kwota > 3000 PLN
- kategoria = 'elektronika'
- godzina nocna (dodaj pole `hour` z losową wartością 0–5)


In [3]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    # 5% szansy na transakcję podejrzaną
    if random.random() < 0.05:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.01, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'timestamp': datetime.now().isoformat(),
            'hour': random.randint(0, 5) # nocna godzina
        }
    else:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 3000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'timestamp': datetime.now().isoformat(),
            'hour': random.randint(6, 23) # dzienna godzina
        }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | Hour: {tx['hour']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


# Część 2: Konsument bezstanowy — filtrowanie (15 min)


## Zadanie 2.1 — Wyświetl duże transakcje

Napisz konsumenta, który wypisuje tylko transakcje z `amount > 1000` (jako ALERT).

In [4]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Uruchomiono konsumenta filtrującego duże transakcje (>1000)...\n")
for message in consumer:
    tx = message.value
    if tx.get('amount', 0) > 1000:
        print(f"🚨 ALERT: Duża transakcja! ID: {tx['tx_id']} | Kwota: {tx['amount']:.2f} PLN | Kategoria: {tx['category']} | Sklep: {tx['store']}")

Writing consumer_filter.py


## Zadanie 2.2 — Dodaj poziom ryzyka

Napisz konsumenta, który dla każdej transakcji dodaje pole `risk_level` w zależności od kwoty:
- `amount > 3000` → "HIGH"
- `amount > 1000` → "MEDIUM"
- pozostałe → "LOW"

In [5]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Uruchomiono konsumenta wzbogacającego o poziom ryzyka...\n")
for message in consumer:
    tx = message.value
    amount = tx.get('amount', 0)
    
    if amount > 3000:
        tx['risk_level'] = 'HIGH'
    elif amount > 1000:
        tx['risk_level'] = 'MEDIUM'
    else:
        tx['risk_level'] = 'LOW'
        
    print(f"ID: {tx['tx_id']} | Kwota: {amount:.2f} PLN | Sklep: {tx['store']} | Ryzyko: {tx['risk_level']}")

Writing consumer_enrich.py


# Część 3: Konsument stanowy — zliczanie (15 min)

## Zadanie 3.1 — Transakcje per sklep

Napisz konsumenta, który liczy transakcje i sumy per sklep. Wypisz podsumowanie co 10 wiadomości.

In [6]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

print("Uruchomiono konsumenta zliczającego statystyki per sklep...\n")
for message in consumer:
    tx = message.value
    store = tx.get('store', 'Nieznany')
    amount = tx.get('amount', 0.0)
    
    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1
    
    if msg_count % 10 == 0:
        print("\n" + "="*60)
        print(f" PODSUMOWANIE (Suma wiadomości: {msg_count})")
        print("="*60)
        print(f" {'Sklep':<15} | {'Liczba transakcji':<18} | {'Łączny obrót (PLN)':<18}")
        print("-"*60)
        for s in sorted(store_counts.keys()):
            print(f" {s:<15} | {store_counts[s]:<18} | {total_amount[s]:<18.2f}")
        print("="*60 + "\n")

Writing consumer_count.py


# Część 4: Reguły decyzyjne (20 min)

## Zadanie 4.1 — Scoring transakcji

Zaimplementuj funkcję `score_transaction(tx)`, która analizuje transakcję i przyznaje punkty karne za podejrzane cechy wg reguł:

| ID | Reguła | Warunek | Punkty |
|----|--------|---------|--------|
| **R1** | Kwota | `amount > 3000` | **+3** |
| **R2** | Kategoria i kwota | `category == 'elektronika'` i `amount > 1500` | **+2** |
| **R3** | Godzina nocna | `hour < 6` | **+2** |

Jeśli suma punktów wynosi **3 lub więcej**, transakcja jest oznaczana jako **PODEJRZANA** (fraud).

In [7]:
from datetime import datetime

def score_transaction(tx):
    score = 0
    rules = []
    
    # R1: amount > 3000
    if tx.get('amount', 0) > 3000:
        score += 3
        rules.append('R1 (amount > 3000)')
        
    # R2: elektronika i amount > 1500
    if tx.get('category') == 'elektronika' and tx.get('amount', 0) > 1500:
        score += 2
        rules.append('R2 (elektronika & amount > 1500)')
        
    # R3: godzina < 6
    hour = tx.get('hour')
    if hour is None and 'timestamp' in tx:
        try:
            dt = datetime.fromisoformat(tx['timestamp'])
            hour = dt.hour
        except:
            pass
            
    if hour is not None and hour < 6:
        score += 2
        rules.append('R3 (hour < 6)')
        
    return score, rules

# Testy
test_tx1 = {'tx_id': 'TX999', 'amount': 4500.0, 'category': 'elektronika', 'timestamp': '2026-04-01T03:15:00', 'hour': 3}
test_tx2 = {'tx_id': 'TX101', 'amount': 150.0, 'category': 'odzież', 'timestamp': '2026-04-01T12:00:00', 'hour': 12}

score1, rules1 = score_transaction(test_tx1)
score2, rules2 = score_transaction(test_tx2)

print(f"Transakcja testowa 1 (Podejrzana): Score = {score1}, Aktywowane reguły = {rules1}")
print(f"Transakcja testowa 2 (Normalna): Score = {score2}, Aktywowane reguły = {rules2}")

Transakcja testowa 1 (Podejrzana): Score = 7, Aktywowane reguły = ['R1 (amount > 3000)', 'R2 (elektronika & amount > 1500)', 'R3 (hour < 6)']
Transakcja testowa 2 (Normalna): Score = 0, Aktywowane reguły = []


## Zadanie 4.2 — Konsument scoringowy

Połącz funkcję scoringu z konsumentem Kafki. Jeżeli transakcja ma łączny score >= 3, prześlij ją na nowy temat `alerts` w Kafce.

Przed uruchomieniem skryptu utwórz temat `alerts` w terminalu kontenera JupyterLab:
```bash
kafka-topics.sh --create --topic alerts --bootstrap-server broker:9092 --partitions 1 --replication-factor 1
```

In [8]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def score_transaction(tx):
    score = 0
    rules = []
    
    if tx.get('amount', 0) > 3000:
        score += 3
        rules.append('R1 (amount > 3000)')
        
    if tx.get('category') == 'elektronika' and tx.get('amount', 0) > 1500:
        score += 2
        rules.append('R2 (elektronika & amount > 1500)')
        
    hour = tx.get('hour')
    if hour is None and 'timestamp' in tx:
        try:
            dt = datetime.fromisoformat(tx['timestamp'])
            hour = dt.hour
        except:
            pass
            
    if hour is not None and hour < 6:
        score += 2
        rules.append('R3 (hour < 6)')
        
    return score, rules

print("Uruchomiono konsumenta scoringowego...\n")
for message in consumer:
    tx = message.value
    score, rules = score_transaction(tx)
    
    if score >= 3:
        alert = {
            'tx_id': tx.get('tx_id'),
            'amount': tx.get('amount'),
            'store': tx.get('store'),
            'category': tx.get('category'),
            'hour': tx.get('hour'),
            'fraud_score': score,
            'triggered_rules': rules,
            'timestamp': datetime.now().isoformat()
        }
        alert_producer.send('alerts', value=alert)
        print(f"🚨 [ALERT] Transakcja {tx['tx_id']} jest podejrzana! Score: {score} | Reguły: {rules} | Kwota: {tx['amount']:.2f} PLN | Godzina: {tx['hour']}")

Writing scoring_consumer.py


# Praca domowa

1. Uruchom jednocześnie:
   - producenta: `python producer.py`
   - filtrowanie: `python consumer_filter.py`
   - scoring: `python scoring_consumer.py`
2. W osobnym terminalu sprawdź temat `alerts` za pomocą wbudowanego CLI Kafki:
   ```bash
   kafka-console-consumer.sh --bootstrap-server broker:9092 --topic alerts --from-beginning
   ```
3. Wypchnij wykonane zadania (kod notebooków i plików skryptów) do swojego repozytorium Git.